# PharmaSpot Morocco — run the API in Google Colab

This notebook **installs the backend**, starts **FastAPI**, and gives you a **public HTTPS link** (via ngrok) so you can point **Vercel** at it with `VITE_API_BASE_URL`.

**Limits (be honest):**
- Colab **disconnects** when idle or after a time limit; the link **stops working** until you re-run cells.
- Free ngrok URLs **change each session** unless you add an [ngrok auth token](https://dashboard.ngrok.com/get-started/your-authtoken).
- First analysis can be **slow** (Overpass + WorldPop); heavy cities may **run out of RAM** on free Colab.
- The **React map UI** is not in this notebook — deploy the `frontend/` on **Vercel** and set `VITE_API_BASE_URL` to the ngrok URL below.


## 1) Clone the project

Change `REPO` if you use a fork.

In [ ]:
import shutil
import subprocess

REPO = "https://github.com/adilchetouani-rgb/pharmacie.git"
BRANCH = "main"

shutil.rmtree("/content/pharmacie", ignore_errors=True)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/pharmacie"],
    check=True,
)
%cd /content/pharmacie

## 2) System libraries (GDAL / spatialindex) + Python packages

This can take **several minutes**.

In [ ]:
%%capture cap --no-stderr
!apt-get update -qq
!apt-get install -y -qq gdal-bin libgdal-dev libspatialindex-dev g++
!pip install -q pyngrok
!pip install -q -r /content/pharmacie/backend/requirements.txt

## 3) (Optional) ngrok auth token — steadier tunnels

1. Sign up at [ngrok](https://ngrok.com/) and copy your **authtoken**.
2. In Colab: **Secrets** (key icon) → add secret `NGROK_AUTHTOKEN` with that value.
3. Or paste the token in the next cell (less safe if you share the notebook).

In [ ]:
import os

try:
    from google.colab import userdata
    token = userdata.get("NGROK_AUTHTOKEN", "")
except Exception:
    token = ""

# Fallback: set here only for private testing (do not commit real tokens)
if not token:
    token = os.environ.get("NGROK_AUTHTOKEN", "")

if token:
    from pyngrok import ngrok
    ngrok.set_auth_token(token.strip())
    print("ngrok authtoken set.")
else:
    print("No NGROK_AUTHTOKEN — anonymous tunnel (URL may change often).")

## 4) Start the API (background) + public URL

Run this once. If you change code, **restart runtime** and re-run from the top.

In [ ]:
import os
import subprocess
import time
import sys

# Stop previous uvicorn if re-running
!pkill -f "uvicorn main:app" 2>/dev/null || true
time.sleep(1)

os.chdir("/content/pharmacie/backend")
sys.path.insert(0, "/content/pharmacie/backend")

log = open("/content/uvicorn.log", "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log,
    stderr=subprocess.STDOUT,
    cwd="/content/pharmacie/backend",
)
time.sleep(3)

from pyngrok import ngrok
ngrok.kill()
tunnel = ngrok.connect(8000)
BASE = getattr(tunnel, "public_url", str(tunnel)).rstrip("/")
print("\n=== Backend base URL (use for Vercel VITE_API_BASE_URL) ===")
print(BASE)
print("\nTest:", BASE + "/api/health")
print("\nKeep this tab open; Colab stops when disconnected.")

## 5) Quick test (optional)

In [ ]:
import urllib.request
import json

with urllib.request.urlopen(BASE + "/api/health", timeout=30) as r:
    print(json.loads(r.read().decode()))

## Vercel

Set **`VITE_API_BASE_URL`** to the printed **`BASE`** (same as ngrok HTTPS URL), then redeploy the frontend.

---

**If install fails:** open the captured pip output by changing the install cell to remove `%%capture`, re-run, and read errors. Some Colab runtimes need a **Runtime → Restart session** after apt installs.